# GPT Model, Decoder-Only Transformer Architecture.

## Goal

Given a sequence of input tokens, generate the very next token in the sequence. 

We can invoke this one-token generation capability autoregressively to generate as many tokens as we want i.e appending the newly generated token to the input sequence and using the extended sequence to predict the next token.

<img src="images/gpt-model-overview.png" width="40%"/>

## Recipe
1. Draft GPT Model (decoder only transformer) design decisions as configuration.
2. Codify the `DummyGPTModel` to understand the overall structure and shape of ins and outs.
3. Invoke an example context (with tokenization) into `DummyGPTModel`.
4. Integrate already prepared components: `TransformerBlock` and Token Embeddings and Positional Embeddings.
5. Invoke the real `GPTModel`.
6. Codify autoregressive text generation application using `GPTModel`.

## Step 1: Draft GPT Model Design Decisions as Configuration

In [1]:
GPT_CONFIG = {
    "vocab_size": 50257,          # Vocabulary size, the number of unique tokens available to be generated.
    "context_window_size": 1024,  # Context window size, the maximum number of tokens model can see, learn and effectively predict on.
    "emb_dim": 768,               # Embedding dimension to capture meaning in a vector space.
    "n_heads": 12,                # Number of attention heads.
    "n_layers": 12,               # Number of transformer block layers. 
    "drop_rate": 0.1,             # Dropout rate to avoid overfitting.
    "qkv_bias": False             # Weather Query-Key-Value matrices includes bias along with weights.
}

## Step 2: Codify a `DummyGPTModel`

- **Input:** A matrix of shape **`[1, 4]`**, representing a batch of **1 sequence** having **4 tokens**.
- **Output:** A matrix of shape **`[1, 4, 50257]`**, representing a batch of **1 sequence** having **4 tokens** and each is associated with **50,257 output scores**—one for every token in the vocabulary. These scores indicate how likely each vocabulary token is to be the **next token** in the sequence.

**Note:** The row **output[3]** contains the scores for all **50,257** vocabulary tokens representing the model's prediction of the token that should appear **after the 4th input token**. These last index as in this case 3 is selected because this represents the very next token associated with the last token in the input sequence of size 4.

<img src="images/gpt-model-internals.png" width="40%"/>


In [2]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) ## projecting vocabulary to embeddings dimensional space.
        self.pos_emb = nn.Embedding(cfg["context_window_size"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg)
              for _ in range(cfg["n_layers"])]
        )
        
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False) ## projecting embeddings to vocabulary_size dimensional space.

    def forward(self, in_idx):
        seq_len = len(in_idx)
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        ## represents the scores for all tokens in the vocabulary, higher score implies higher chances of being next token.
        logits = self.out_head(x)
        
        return logits

class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x

class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x

**Note:** _Some components are placeholders for now like `DummyTransformerBlock` and `DummyLayerNorm`. They will be replaced with real ones in real implementation._

## Step 3: Invoke an Example Context (with tokenization) into `DummyGPTModel`

In [3]:
import tiktoken

context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
tokenized_context = tokenizer.encode(context)
input_sequence = torch.tensor(tokenized_context).unsqueeze(0)

gpt_model = DummyGPTModel(cfg = GPT_CONFIG)
output = gpt_model(input_sequence)

print(f"Shape of input sequence from GPT: {input_sequence.shape}")
print(f"Shape of output sequence from GPT: {output.shape}")

Shape of input sequence from GPT: torch.Size([1, 4])
Shape of output sequence from GPT: torch.Size([1, 4, 50257])


**Notice:** 
- Shape of input sequence: **`[1, 4]`**, representing a batch of **1 sequence** having **4 tokens**.
- Shape of output sequence: **`[1, 4, 50257]`**, representing a batch of **1 sequence** having **4 tokens** and each is associated with **50,257 output scores**—one for every token in the vocabulary. These scores indicate how likely each vocabulary token is to be the **next token** in the sequence.

## Step 4: Integrate Already Prepared Components: Transformer Block and Token Embeddings and Positional Embeddings

There are two separate embeddings for token embeddings and positional embeddings to make the input embeddings position aware.

We have discussed embeddings in detail in one of the previous [computational essay](https://github.com/umairkhancis/llm-under-the-hood/blob/main/embeddings_module/notebooks/tokenization-embedding-essay.ipynb).

In [14]:
import torch
import torch.nn as nn

# Import already prepared `TransformerBlock` abstraction
import sys 
sys.path.append("../..")
from transformer_module import TransformerBlock
from transformer_module import LayerNorm

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) ## projecting vocabulary to embeddings dimensional space.
        self.pos_emb = nn.Embedding(cfg["context_window_size"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False) ## projecting embeddings to vocabulary_size dimensional space.

    def forward(self, in_idx):
        batch, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        ## represents the scores for all tokens in the vocabulary, higher score implies higher chances of being next token.
        logits = self.out_head(x)
        
        return logits

## Step 5: Invoke the real `GPTModel`

In [15]:
import tiktoken

context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
tokenized_context = tokenizer.encode(context)
input_sequence = torch.tensor(tokenized_context).unsqueeze(0)

gpt_model = DummyGPTModel(cfg = GPT_CONFIG)
output = gpt_model(input_sequence)

print(f"Shape of input sequence from GPT: {input_sequence.shape}")
print(f"Shape of output sequence from GPT: {output.shape}")

Shape of input sequence from GPT: torch.Size([1, 4])
Shape of output sequence from GPT: torch.Size([1, 4, 50257])


## Step 6: Codify Autoregressive Text Generation Application using `GPTModel`

The `GPTModel` has a single responsibility: given an input sequence, predict the scores of every vocabulary token being the **next token** at every position in the sequence.

The **application layer** has a different responsibility: orchestrating the autoregressive text generation process by:

1. Tokenizing the input text.
2. Truncating the input sequence to the context window size, if necessary.
3. Invoking the `GPTModel`.
4. Selecting the scores corresponding to the **last token** in the input sequence.
5. Converting the scores into probabilities.
6. Selecting the next token ID.
7. Appending the predicted token ID to the input sequence.
8. Repeating the process to generate subsequent tokens autoregressively.
9. Decoding the final token sequence back into text.

This clear separation of responsibilities keeps the **`GPTModel`** focused solely on prediction, while the **application layer** manages the iterative generation workflow.

<img src="images/text-generation-app.png" width="50%"/>

In [16]:
def text_generation_app(model, input_text, max_new_tokens, context_window_size):
    
    for i in range(max_new_tokens):
        # Truncate input_text to the size limit of the context window.
        limited_context_window = input_text[:, -context_window_size:]

        # Disable computation graph of back propgation in background
        with torch.no_grad():
            logits = model(limited_context_window)

        # Extract the last index value
        logits = logits[:, -1, :]

        # Normalize for easier interpretation of logits scores as probabilities; it is optional as high scores in logits always leads to high probablities.
        probabilities = torch.softmax(logits, dim=-1)

        # Extract the index of the max value in probabilities as index represent the most likelable next_token_id which should come after the last token in the input_text.
        next_token_id = torch.argmax(probabilities, dim=-1, keepdim=True)

        # Concatenate next_token_id to the input_text.
        input_text = torch.cat((input_text, next_token_id), dim=1)

    # At the end of the loop (reaching limit of max tokens) autoregressiveness of input_text is the output_text.
    output_text = input_text
    
    return output_text

### Using `text_generation_app` for one token only.

<img src="images/next-token-app.png" width="30%"/>

In [17]:
torch.manual_seed(123)

context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
tokenized_context = tokenizer.encode(context)
input_sequence = torch.tensor(tokenized_context).unsqueeze(0)

output = text_generation_app(
    model=gpt_model,
    input_text=input_sequence, 
    max_new_tokens=1,
    context_window_size=GPT_CONFIG["context_window_size"]
)
decoded_text = tokenizer.decode(output.squeeze(0).tolist())

print(f"Input: {context}\n")
print(f"Bot: {decoded_text}\n")

Input: Every effort moves you

Bot: Every effort moves you Bits



**Note:** _The next token generated for the context `"Every effort moves you"` is not `"forward"` because the model has not been trained yet. At this stage, it contains only randomly initialized weights, so it has not yet learned the statistical patterns of language required to generate coherent text._

### Using `text_generation_app` autoregressively for maximum 100 new tokens.

In [18]:
torch.manual_seed(123)

context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
tokenized_context = tokenizer.encode(context)
input_sequence = torch.tensor(tokenized_context).unsqueeze(0)

output = text_generation_app(
    model=gpt_model,
    input_text=input_sequence, 
    max_new_tokens=100,
    context_window_size=GPT_CONFIG["context_window_size"]
)
decoded_text = tokenizer.decode(output.squeeze(0).tolist())

print(f"Input: {context}\n")
print(f"Bot: {decoded_text}\n")

Input: Every effort moves you

Bot: Every effort moves you Bits honesty RebeccaSSLic SOerieonce PedJs Fredpps Control stems Bray hoverholmbettOG Marketing Animated goes light Animated goes light dealers XX Berman siege ejectStaffatching challengersETA resolved science vit honesty Rebecca pitfallsatching eject rhy]=Script 63 declining icingigr stitches concealedintend depressalde unpredictable delicate Controlpathic playoffs div picks picks picksEphours bruteOG SO Sketch .ceptionsiddyumapathicrole store Lun Ragnarok ejectウス cynical fire Sar Albion helpless constitutespeg SJ stemsaley stems curses Foxaution Vie695 Directive██icon



**Note:** _The next token generated for the context `"Every effort moves you"` is coherent at all because the model has not been trained yet. At this stage, it contains only randomly initialized weights, so it has not yet learned the statistical patterns of language required to generate coherent text._

## What Did We Learn?

In this computational essay, we reconstructed the **GPT architecture** by assembling the abstractions that we have implemented throughout the previous computational essays.

We learned:

* How the engineering design decisions of a GPT model are captured in a configuration object.
* How a GPT model transforms an input sequence of token IDs into output scores over the entire vocabulary.
* How token embeddings, positional embeddings, and Transformer blocks work together to process an input sequence.
* How the GPT model is responsible only for predicting the next-token scores, while the application layer is responsible for orchestrating autoregressive text generation.
* How autoregressive generation repeatedly feeds the newly generated token back into the input sequence to generate arbitrarily long text.

At this point, we have reconstructed the complete inference pipeline of a GPT model, from an input text sequence to autoregressive text generation.

The only missing piece is **learning**. So far, every weight in the GPT model has been randomly initialized. In the next computational essay, we will finally train the model so that it learns the statistical patterns of language and begins generating coherent text.
